# Chennai Metro Station Activity Analysis

## Project Overview

This project performs exploratory data analysis on Chennai Metro
station activity data using Python, Pandas and NumPy.

The analysis focuses on passenger activity, waiting times,
complaints, delays, parking availability and station-level
operational indicators.

## Objectives

- Explore the metro station dataset
- Analyze passenger entries and exits
- Compare stations across zones
- Identify stations with high waiting times and complaints
- Analyze parking availability
- Calculate weekly delays
- Handle missing values
- Rank stations based on daily entries
- Create an exploratory station attention score

In [2]:
import pandas as pd
import numpy as np

## 1. Load the Dataset

In [3]:
file_path = "../data/challenge_dataset_chennai_metro_station_activity.csv"

metro_data = pd.read_csv(file_path)

metro_data.head()

,station_id,station_name,zone,station_type,daily_entries,daily_exits,peak_hour_load_pct,off_peak_load_pct,avg_wait_minutes,weekday_delays,weekend_delays,monthly_complaints,has_parking
0,1,Central,North,Interchange,201,18450,72,58,4.2,9,14,32.0,False
1,2,Egmore,North,Interchange,156,13200,68,51,3.7,7,11,28.0,False
2,3,Shenoy Nagar,Central,Residential,84,7100,41,37,2.1,4,8,16.0,False
3,4,Anna Nagar East,Central,Residential,97,8250,46,42,2.4,5,9,18.0,False
4,5,Koyambedu,West,Commercial,143,12100,63,55,3.5,8,13,26.0,False


## 2. Dataset Overview

Before performing the analysis, we inspect the structure,
dimensions and data types of the dataset.

In [4]:
metro_data.shape

(40, 13)

In [5]:
metro_data.columns

Index(['station_id', 'station_name', 'zone', 'station_type', 'daily_entries',
       'daily_exits', 'peak_hour_load_pct', 'off_peak_load_pct',
       'avg_wait_minutes', 'weekday_delays', 'weekend_delays',
       'monthly_complaints', 'has_parking'],
      dtype='object')

In [ ]:
metro_data.dtypes

station_id              int64
station_name           object
zone                   object
station_type           object
daily_entries           int64
daily_exits             int64
peak_hour_load_pct      int64
off_peak_load_pct       int64
avg_wait_minutes      float64
weekday_delays          int64
weekend_delays          int64
monthly_complaints    float64
has_parking              bool
dtype: object

In [9]:
metro_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   station_id          40 non-null     int64  
 1   station_name        40 non-null     object 
 2   zone                40 non-null     object 
 3   station_type        40 non-null     object 
 4   daily_entries       40 non-null     int64  
 5   daily_exits         40 non-null     int64  
 6   peak_hour_load_pct  40 non-null     int64  
 7   off_peak_load_pct   40 non-null     int64  
 8   avg_wait_minutes    37 non-null     float64
 9   weekday_delays      40 non-null     int64  
 10  weekend_delays      40 non-null     int64  
 11  monthly_complaints  38 non-null     float64
 12  has_parking         40 non-null     bool   
dtypes: bool(1), float64(2), int64(7), object(3)
memory usage: 3.9+ KB


In [10]:
metro_data.describe()

,station_id,daily_entries,daily_exits,peak_hour_load_pct,off_peak_load_pct,avg_wait_minutes,weekday_delays,weekend_delays,monthly_complaints
count,40.000000,40.000000,40.000000,40.000000,40.000000,37.000000,40.000000,40.000000,38.000000
mean,20.500000,117.025000,9910.000000,52.750000,45.625000,2.945946,6.075000,10.250000,21.263158
std,11.690452,36.788019,3520.165352,12.593751,9.752547,0.987588,2.268937,3.127627,6.348799
min,1.000000,59.000000,4700.000000,28.000000,26.000000,1.200000,2.000000,5.000000,10.000000
25%,10.750000,91.750000,7575.000000,43.750000,39.000000,2.300000,4.750000,8.000000,17.000000
50%,20.500000,110.000000,9150.000000,52.500000,45.000000,2.800000,6.000000,10.000000,20.500000
75%,30.250000,134.750000,11575.000000,62.250000,53.250000,3.700000,7.250000,12.000000,24.750000
max,40.000000,219.000000,20100.000000,78.000000,66.000000,5.300000,12.000000,18.000000,39.000000


## 3. Missing Value Analysis

Missing values are checked before performing further analysis.

In [ ]:
metro_data.isnull().sum()

## 4. Station Filtering

### Stations in the South Zone

In [11]:
south_stations = metro_data[
    metro_data['zone'] == 'South'
]

south_stations

,station_id,station_name,zone,station_type,daily_entries,daily_exits,peak_hour_load_pct,off_peak_load_pct,avg_wait_minutes,weekday_delays,weekend_delays,monthly_complaints,has_parking
6,7,Guindy,South,Business,176,15800,71,61,4.6,10,16,31.0,False
7,8,Alandur,South,Interchange,219,20100,78,66,5.3,12,18,39.0,False
8,9,Airport,South,Transport,188,17200,74,63,4.9,11,17,35.0,False
9,10,Little Mount,South,Residential,91,7600,44,39,2.3,5,9,17.0,False
10,11,Saidapet,South,Residential,109,9300,52,44,2.8,6,10,21.0,False
11,12,Nanganallur Road,South,Residential,88,6900,39,35,1.9,4,7,NaN,False
12,13,St. Thomas Mount,South,Interchange,151,12600,61,54,3.8,8,13,27.0,False
37,38,St. Thomas Mount,South,Interchange,151,12600,61,54,3.8,8,13,27.0,False
38,39,Pallavaram,South,Residential,112,9100,50,45,2.7,6,10,21.0,False
39,40,Chromepet,South,Education,99,8100,46,41,2.4,5,9,18.0,False


### Stations with High Passenger Activity and Peak-Hour Load

The following analysis identifies stations with more than 10,000
daily entries and a peak-hour load above 60%.

In [13]:
high_activity_stations = metro_data[
    (metro_data['daily_entries'] > 10000) &
    (metro_data['peak_hour_load_pct'] > 60)
]

high_activity_stations

,station_id,station_name,zone,station_type,daily_entries,daily_exits,peak_hour_load_pct,off_peak_load_pct,avg_wait_minutes,weekday_delays,weekend_delays,monthly_complaints,has_parking


In [14]:
high_priority_stations = metro_data[
    (metro_data['daily_entries'] > 15000) |
    (metro_data['monthly_complaints'] > 30)
]

high_priority_stations

,station_id,station_name,zone,station_type,daily_entries,daily_exits,peak_hour_load_pct,off_peak_load_pct,avg_wait_minutes,weekday_delays,weekend_delays,monthly_complaints,has_parking
0,1,Central,North,Interchange,201,18450,72,58,4.2,9,14,32.0,False
6,7,Guindy,South,Business,176,15800,71,61,4.6,10,16,31.0,False
7,8,Alandur,South,Interchange,219,20100,78,66,5.3,12,18,39.0,False
8,9,Airport,South,Transport,188,17200,74,63,4.9,11,17,35.0,False


## 5. Selecting Specific Columns

The following analysis focuses on Central zone stations and
selected operational indicators.

In [15]:
central_stations = metro_data.loc[
    metro_data['zone'] == 'Central',
    [
        'station_name',
        'zone',
        'daily_entries',
        'avg_wait_minutes'
    ]
]

central_stations

,station_name,zone,daily_entries,avg_wait_minutes
2,Shenoy Nagar,Central,84,2.1
3,Anna Nagar East,Central,97,2.4
13,Teynampet,Central,134,3.9
14,Thousand Lights,Central,119,3.2
15,Nandanam,Central,111,2.9
30,Purasaiwakkam,Central,113,2.9
31,Kilpauk,Central,86,2.0
32,Chetpet,Central,104,2.6
33,Nungambakkam,Central,127,3.5


### Station with the Longest Average Waiting Time

In [16]:
longest_waiting_station = metro_data.loc[
    metro_data['avg_wait_minutes'].idxmax(),
    'station_name'
]

longest_waiting_station

'Alandur'

The station identified above has the highest recorded average
waiting time in the dataset.

## 6. Parking Availability

Stations are classified according to whether parking is available.

In [17]:
no_parking = metro_data.loc[
    metro_data['has_parking'] == False,
    'station_name'
]

no_parking

0               Central
1                Egmore
2          Shenoy Nagar
3       Anna Nagar East
4             Koyambedu
5            Vadapalani
6                Guindy
7               Alandur
8               Airport
9          Little Mount
10             Saidapet
11     Nanganallur Road
12     St. Thomas Mount
13            Teynampet
14      Thousand Lights
15             Nandanam
16         Washermenpet
17              Mannadi
18           High Court
19          Wimco Nagar
20    Wimco Nagar Depot
21     New Washermenpet
22          Neelankarai
23                Adyar
24         Besant Nagar
25               Marina
26             Mylapore
27           Triplicane
28           Royapettah
29             Perambur
30        Purasaiwakkam
31              Kilpauk
32              Chetpet
33         Nungambakkam
34          Kodambakkam
35          Ashok Nagar
36        Ekkatuthangal
37     St. Thomas Mount
38           Pallavaram
39            Chromepet
Name: station_name, dtype: object

In [18]:
parking_count = (metro_data['has_parking'] == True).sum()
no_parking_count = (metro_data['has_parking'] == False).sum()

print("Stations with parking:", parking_count)
print("Stations without parking:", no_parking_count)

Stations with parking: 0
Stations without parking: 40


## 7. Weekly Delay Analysis

Weekday and weekend delays are combined to calculate total weekly
delays for each station.

In [19]:
metro_copy = metro_data.copy()

metro_copy['total_weekly_delays'] = (
    metro_copy['weekday_delays'] +
    metro_copy['weekend_delays']
)

metro_copy[
    ['station_name', 'total_weekly_delays']
]

,station_name,total_weekly_delays
0,Central,23
1,Egmore,18
2,Shenoy Nagar,12
3,Anna Nagar East,14
4,Koyambedu,21
5,Vadapalani,19
6,Guindy,26
7,Alandur,30
8,Airport,28
9,Little Mount,14


## 8. Data Cleaning

Missing values in average waiting time and monthly complaints are
handled using mean and median imputation respectively.

In [20]:
metro_copy = metro_data.copy()

average_waiting_time = metro_copy['avg_wait_minutes'].mean()

metro_copy['avg_wait_minutes'] = (
    metro_copy['avg_wait_minutes']
    .fillna(average_waiting_time)
)

median_complaints = metro_copy['monthly_complaints'].median()

metro_copy['monthly_complaints'] = (
    metro_copy['monthly_complaints']
    .fillna(median_complaints)
)

In [21]:
metro_copy[
    ['avg_wait_minutes', 'monthly_complaints']
].isnull().sum()

avg_wait_minutes      0
monthly_complaints    0
dtype: int64

## 9. Zone-Level Analysis

Station performance is compared across zones using grouped
statistics.

In [22]:
average_waiting_time_by_zone = (
    metro_copy
    .groupby('zone')['avg_wait_minutes']
    .mean()
)

average_waiting_time_by_zone

zone
Central    2.833333
East       2.971429
North      2.699099
South      3.450000
West       2.549189
Name: avg_wait_minutes, dtype: float64

In [23]:
max_daily_entries_zone = (
    metro_copy
    .groupby('zone')['daily_entries']
    .max()
)

max_daily_entries_zone

zone
Central    134
East       163
North      201
South      219
West       143
Name: daily_entries, dtype: int64

In [24]:
number_of_stations = (
    metro_copy
    .groupby('zone')['station_name']
    .count()
)

number_of_stations

zone
Central     9
East        7
North       9
South      10
West        5
Name: station_name, dtype: int64

In [25]:
zone_summary = (
    metro_copy
    .groupby('zone')
    .agg(
        number_of_stations=('station_name', 'count'),
        average_entries=('daily_entries', 'mean'),
        maximum_entries=('daily_entries', 'max'),
        minimum_entries=('daily_entries', 'min'),
        average_waiting_time=('avg_wait_minutes', 'mean'),
        total_complaints=('monthly_complaints', 'sum')
    )
    .reset_index()
)

zone_summary

,zone,number_of_stations,average_entries,maximum_entries,minimum_entries,average_waiting_time,total_complaints
0,Central,9,108.333333,134,84,2.833333,181.0
1,East,7,116.714286,163,74,2.971429,141.5
2,North,9,107.000000,201,59,2.699099,171.0
3,South,10,138.400000,219,88,3.450000,256.5
4,West,5,108.400000,143,78,2.549189,99.0


## 10. NumPy Analysis

NumPy is used to perform numerical calculations on passenger entry
and exit data.

In [26]:
entries_exit_array = metro_copy[
    ['daily_entries', 'daily_exits']
].to_numpy()

entries_exit_array.shape

(40, 2)

In [27]:
entry = metro_copy['daily_entries'].to_numpy()
exit = metro_copy['daily_exits'].to_numpy()

In [28]:
difference = entry - exit

difference

array([-18249, -13044,  -7016,  -8153, -11957, -10772, -15624, -19881,
       -17012,  -7509,  -9191,  -6812, -12449, -11366,  -9981,  -9289,
        -8498,  -6127, -10571,  -5333,  -4641,  -6419,  -5726,  -9584,
        -7802, -13937, -11663, -10179,  -8792,  -7705,  -9087,  -7114,
        -8696, -10473,  -7408,  -8199,  -6022, -12449,  -8988,  -8001])

In [29]:
max_entry_exit_difference = difference.max()

max_entry_exit_difference

np.int64(-4641)

In [30]:
max_index = np.argmax(difference)

station_with_max_difference = metro_copy.loc[
    max_index,
    ['station_name', 'daily_entries', 'daily_exits']
]

station_with_max_difference

station_name     Wimco Nagar Depot
daily_entries                   59
daily_exits                   4700
Name: 20, dtype: object

In [31]:
max_difference_position = np.argmax(difference)

station_with_max_difference = metro_copy.iloc[
    max_difference_position
][
    ['station_name', 'daily_entries', 'daily_exits']
]

station_with_max_difference

station_name     Wimco Nagar Depot
daily_entries                   59
daily_exits                   4700
Name: 20, dtype: object

### Overall Entry-to-Exit Ratio

The total number of daily entries is compared with the total number
of daily exits across all stations.

In [32]:
total_entries = entry.sum()
total_exits = exit.sum()

entry_exit_ratio = total_entries / total_exits

entry_exit_ratio

np.float64(0.011808779011099899)

In [33]:
print("Total entries:", total_entries)
print("Total exits:", total_exits)
print("Entry-exit ratio:", entry_exit_ratio)

Total entries: 4681
Total exits: 396400
Entry-exit ratio: 0.011808779011099899


## 11. Busiest Stations

The top 10 stations are identified based on daily passenger entries.

In [34]:
busiest_10_stations = (
    metro_copy
    .sort_values('daily_entries', ascending=False)
    .head(10)
)

busiest_10_stations[
    ['station_name', 'daily_entries']
]

,station_name,daily_entries
7,Alandur,219
0,Central,201
8,Airport,188
6,Guindy,176
25,Marina,163
1,Egmore,156
12,St. Thomas Mount,151
37,St. Thomas Mount,151
4,Koyambedu,143
26,Mylapore,137


## 12. Stations Requiring Attention

Stations with more than 20 monthly complaints and an average
waiting time above 3 minutes are identified for further analysis.

In [36]:
problem_stations = metro_copy.loc[
    (metro_copy['monthly_complaints'] > 20) &
    (metro_copy['avg_wait_minutes'] > 3),
    [
        'station_name',
        'monthly_complaints',
        'avg_wait_minutes'
    ]
]

problem_stations

,station_name,monthly_complaints,avg_wait_minutes
0,Central,32.0,4.2
1,Egmore,28.0,3.7
4,Koyambedu,26.0,3.5
6,Guindy,31.0,4.6
7,Alandur,39.0,5.3
8,Airport,35.0,4.9
12,St. Thomas Mount,27.0,3.8
13,Teynampet,25.0,3.9
14,Thousand Lights,22.0,3.2
18,High Court,24.0,3.6


## 13. Station Ranking

Stations are ranked according to daily passenger entries, with the
station having the highest number of entries receiving rank 1.

In [37]:
metro_copy['entry_rank'] = (
    metro_copy['daily_entries']
    .rank(method='first', ascending=False)
    .astype(int)
)

In [38]:
entry_ranked_stations = (
    metro_copy
    .sort_values('entry_rank')
)

entry_ranked_stations[
    ['station_name', 'daily_entries', 'entry_rank']
]

,station_name,daily_entries,entry_rank
7,Alandur,219,1
0,Central,201,2
8,Airport,188,3
6,Guindy,176,4
25,Marina,163,5
1,Egmore,156,6
12,St. Thomas Mount,151,7
37,St. Thomas Mount,151,8
4,Koyambedu,143,9
26,Mylapore,137,10


## 14. Zone Performance Summary

The following table summarizes passenger activity, waiting time and
complaints across different zones.

In [39]:
zone_summary = (
    metro_copy
    .groupby('zone')
    .agg(
        number_of_stations=('station_name', 'count'),
        average_entries=('daily_entries', 'mean'),
        average_exits=('daily_exits', 'mean'),
        average_waiting_time=('avg_wait_minutes', 'mean'),
        total_complaints=('monthly_complaints', 'sum')
    )
    .reset_index()
)

zone_summary

,zone,number_of_stations,average_entries,average_exits,average_waiting_time,total_complaints
0,Central,9,108.333333,9127.777778,2.833333,181.0
1,East,7,116.714286,9785.714286,2.971429,141.5
2,North,9,107.000000,9061.111111,2.699099,171.0
3,South,10,138.400000,11930.000000,3.450000,256.5
4,West,5,108.400000,8980.000000,2.549189,99.0


## 15. Exploratory Station Attention Score

An exploratory attention score is created by combining the relative
rankings of:

- Daily passenger entries
- Monthly complaints
- Average waiting time

Higher scores indicate stations that rank relatively higher across
these three indicators.

This is a custom exploratory metric and is not an official metro
operational measure.

In [40]:
attention = metro_copy.copy()

In [41]:
attention['entries_rank'] = (
    attention['daily_entries']
    .rank(ascending=True)
)

attention['complaints_rank'] = (
    attention['monthly_complaints']
    .rank(ascending=True)
)

attention['waiting_rank'] = (
    attention['avg_wait_minutes']
    .rank(ascending=True)
)

In [42]:
attention['attention_score'] = (
    attention['entries_rank'] +
    attention['complaints_rank'] +
    attention['waiting_rank']
)

In [43]:
most_attention = (
    attention
    .sort_values('attention_score', ascending=False)
    .head(10)
)

most_attention[
    [
        'station_name',
        'zone',
        'daily_entries',
        'avg_wait_minutes',
        'monthly_complaints',
        'attention_score'
    ]
]

,station_name,zone,daily_entries,avg_wait_minutes,monthly_complaints,attention_score
7,Alandur,South,219,5.3,39.0,120.0
8,Airport,South,188,4.9,35.0,116.0
0,Central,North,201,4.2,32.0,114.0
6,Guindy,South,176,4.6,31.0,112.0
1,Egmore,North,156,3.7,28.0,102.0
37,St. Thomas Mount,South,151,3.8,27.0,101.0
12,St. Thomas Mount,South,151,3.8,27.0,101.0
26,Mylapore,East,137,3.8,26.0,96.5
13,Teynampet,Central,134,3.9,25.0,96.0
4,Koyambedu,West,143,3.5,26.0,93.0


# 16. Key Findings

The analysis explored passenger activity, waiting times, complaints,
parking availability and delays across Chennai Metro stations.

### Passenger Activity

- The top 10 busiest stations were identified based on daily entries.
- Entry and exit volumes were compared across stations.

### Waiting Time

- Average waiting time was compared across zones.
- Stations with higher waiting times were identified.

### Complaints

- Stations with more than 20 monthly complaints and average waiting
  time above 3 minutes were identified for further investigation.

### Parking

- Stations were classified according to parking availability.

### Delays

- Weekday and weekend delays were combined to calculate total weekly
  delays.

### Station Ranking

- Stations were ranked according to daily passenger entries.

### Exploratory Attention Score

- A custom score combining entry, complaint and waiting-time rankings
  was created as an exploratory indicator for identifying stations
  that may require further investigation.